In [3]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from numpy import pi
import numpy as np

In [4]:
def build_cuccaro_circuit(a, b):
    qreg_q = QuantumRegister(10, 'q')
    creg_c = ClassicalRegister(5, 'c')
    circuit = QuantumCircuit(qreg_q, creg_c)

    # q1,q4,q7,q10 = bits of A (LSB to MSB)
    # q2,q5,q8,q11 = bits of B (LSB to MSB)

    # encode A
    if a & 1: circuit.x(qreg_q[1])
    if a & 2: circuit.x(qreg_q[4])
    if a & 4: circuit.x(qreg_q[6])
    if a & 8: circuit.x(qreg_q[8])

    # encode B
    if b & 1: circuit.x(qreg_q[0])
    if b & 2: circuit.x(qreg_q[3])
    if b & 4: circuit.x(qreg_q[5])
    if b & 8: circuit.x(qreg_q[7])

    circuit.barrier()

    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.cx(qreg_q[8], qreg_q[9])
    circuit.barrier(qreg_q[3])
    circuit.x(qreg_q[3])
    circuit.ccx(qreg_q[6], qreg_q[7], qreg_q[9])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[2], qreg_q[3])
    circuit.cx(qreg_q[4], qreg_q[5])
    circuit.cx(qreg_q[6], qreg_q[7])
    circuit.ccx(qreg_q[4], qreg_q[5], qreg_q[6])
    circuit.ccx(qreg_q[2], qreg_q[3], qreg_q[4])
    circuit.x(qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[6])
    circuit.ccx(qreg_q[0], qreg_q[1], qreg_q[2])
    circuit.x(qreg_q[3])
    circuit.cx(qreg_q[6], qreg_q[4])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[4], qreg_q[2])
    circuit.barrier(qreg_q[0])
    circuit.barrier(qreg_q[6])
    circuit.barrier(qreg_q[8])
    circuit.cx(qreg_q[6], qreg_q[5])
    circuit.cx(qreg_q[8], qreg_q[7])
    circuit.cx(qreg_q[1], qreg_q[0])
    circuit.cx(qreg_q[4], qreg_q[3])
    circuit.measure(qreg_q[0], creg_c[0])
    circuit.measure(qreg_q[3], creg_c[1])
    circuit.measure(qreg_q[5], creg_c[2])
    circuit.measure(qreg_q[7], creg_c[3])
    circuit.measure(qreg_q[9], creg_c[4])


    return circuit

In [11]:
shots = 1000

# ====================================================
# DEPOLARIZING NOISE MODEL
# ====================================================

# You will likely need to tune this parameter to match the 37.28% ER from the paper
p = 0.0249

# ----------------------------------------------------
# Depolarizing Errors for 1, 2, and 3 qubits
# ----------------------------------------------------
# Qiskit's built-in function automatically handles the complex math
# of creating symmetric multi-qubit depolarizing channels.

error_1 = depolarizing_error(p, 1)
error_2 = depolarizing_error(p, 2)
error_3 = depolarizing_error(p, 3)

# ====================================================
# CREATE NOISE MODEL
# ====================================================

noise_model = NoiseModel()

# ----------------------------------------------------
# Apply noise to X gates
# ----------------------------------------------------
noise_model.add_all_qubit_quantum_error(
    error_1,
    ['x']
)

# ----------------------------------------------------
# Apply noise to CX gates
# ----------------------------------------------------
noise_model.add_all_qubit_quantum_error(
    error_2,
    ['cx']
)

# ----------------------------------------------------
# Apply noise to CCX gates
# ----------------------------------------------------
noise_model.add_all_qubit_quantum_error(
    error_3,
    ['ccx']
)

sim = AerSimulator(noise_model=noise_model)

# ====================================================
# NUMPY ARRAYS FOR METRICS
# ====================================================

ER_arr = np.zeros((16, 16))
NMED_arr = np.zeros((16, 16))
MRED_arr = np.zeros((16, 16))

print("Running Depolarizing Noise Simulation...")

for a in range(16):
    for b in range(16):

        qc = build_cuccaro_circuit(a, b)

        compiled = transpile(qc, sim)

        counts = sim.run(
            compiled,
            shots=shots
        ).result().get_counts()

        correct = a + b
        correct_output = format(correct, '05b')

        correct_counts = counts.get(correct_output, 0)

        ER = 1 - correct_counts / shots

        total_ED = 0
        total_relative_ED = 0

        for output, freq in counts.items():

            noisy_decimal = int(output, 2)

            ED = abs(correct - noisy_decimal)

            total_ED += ED * freq

            if correct != 0:
                total_relative_ED += (ED / correct) * freq

        mean_ED = total_ED / shots

        D = 30

        NMED = mean_ED / D
        MRED = total_relative_ED / shots

        ER_arr[a, b] = ER
        NMED_arr[a, b] = NMED
        MRED_arr[a, b] = MRED

# ====================================================
# OVERALL METRICS
# ====================================================

er = np.mean(ER_arr)
nmed = np.mean(NMED_arr)
mred = np.mean(MRED_arr)

print("\n===================================")
print("Depolarizing Noise Results")
print("===================================")

print(f"ER (%) : {er * 100:.2f}")
print(f"NMED   : {nmed:.3f}")
print(f"MRED   : {mred:.3f}")

# Flatten to 1D if needed
er_array = ER_arr.flatten()

print("\nER Array:")
print(er_array)

Running Depolarizing Noise Simulation...

Depolarizing Noise Results
ER (%) : 36.75
NMED   : 0.062
MRED   : 0.182

ER Array:
[0.357 0.387 0.352 0.373 0.352 0.367 0.328 0.347 0.368 0.405 0.345 0.362
 0.367 0.415 0.331 0.341 0.355 0.341 0.375 0.381 0.383 0.36  0.356 0.351
 0.361 0.36  0.346 0.351 0.353 0.351 0.335 0.354 0.345 0.348 0.364 0.348
 0.327 0.353 0.344 0.358 0.327 0.337 0.343 0.371 0.336 0.377 0.359 0.35
 0.374 0.381 0.361 0.409 0.318 0.357 0.367 0.359 0.348 0.397 0.352 0.398
 0.336 0.374 0.368 0.38  0.353 0.359 0.373 0.341 0.409 0.39  0.358 0.373
 0.383 0.39  0.338 0.37  0.402 0.377 0.378 0.356 0.347 0.363 0.338 0.366
 0.379 0.38  0.374 0.363 0.363 0.37  0.367 0.352 0.387 0.379 0.387 0.357
 0.335 0.365 0.348 0.399 0.354 0.347 0.371 0.391 0.322 0.354 0.359 0.374
 0.347 0.374 0.38  0.387 0.356 0.356 0.371 0.379 0.362 0.392 0.391 0.416
 0.35  0.353 0.346 0.381 0.349 0.399 0.406 0.417 0.382 0.387 0.313 0.35
 0.367 0.368 0.333 0.354 0.392 0.371 0.377 0.365 0.395 0.39  0.375 0.365
 